# LC-ORF Bootstrap, Alert Budget, and Audit Profile

This notebook is the next novelty-upgrade step after exporting raw predictions.

It consumes prediction-export CSVs from milestones 03, 06, 07, 08, and 09, then writes:

- paired or stratified bootstrap uncertainty intervals;
- fixed-alert-budget operational metrics;
- a multidimensional LC-ORF audit profile;
- 600-DPI figures for the paper.

Important: the notebook reports whether each comparison is truly `paired` or `stratified_unpaired`. Do not hide that distinction in the paper.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import json
import math
import os
import re
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
)

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 180)

## 1. Configuration

Use exact mode for final paper evidence. Use smoke mode only to test that the pipeline runs.

In [ ]:
PROJECT_ROOT = Path('/content/drive/MyDrive/Credit__')
FORMALIZATION_DIR = PROJECT_ROOT / '10_framework_formalization'
RESULTS_DIR = FORMALIZATION_DIR / 'results'
FIGURES_DIR = FORMALIZATION_DIR / 'figures'
LOGS_DIR = FORMALIZATION_DIR / 'logs'
CHECKPOINT_DIR = FORMALIZATION_DIR / 'bootstrap_checkpoints'
ALERT_CHECKPOINT_DIR = FORMALIZATION_DIR / 'alert_budget_checkpoints'

for path in [FORMALIZATION_DIR, RESULTS_DIR, FIGURES_DIR, LOGS_DIR, CHECKPOINT_DIR, ALERT_CHECKPOINT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SMOKE_TEST = False
N_BOOTSTRAP = 50 if SMOKE_TEST else 1000
RANDOM_SEED = 20260924

# Final paper setting should be None. If Colab is too slow, set e.g. 50000 and report it honestly.
MAX_BOOTSTRAP_ROWS_PER_CLASS = None

METRICS = [
    'average_precision',
    'mcc',
    'precision',
    'recall',
    'brier',
    'ece',
    'cost_per_10k',
]

ALERT_BUDGETS_PER_10K = [100, 500, 1000]
TOP_FRACTION_BUDGETS = [0.01]

FP_COST = 1
FN_COST = 100

PRACTICAL_THRESHOLDS = {
    'average_precision': 0.020,
    'mcc': 0.050,
    'precision': 0.050,
    'recall': 0.050,
    'brier': 0.005,
    'ece': 0.020,
    'cost_per_10k': None,
    'alert_precision': 0.050,
    'alert_recall': 0.050,
}

print('PROJECT_ROOT:', PROJECT_ROOT)
print('N_BOOTSTRAP:', N_BOOTSTRAP)
print('MAX_BOOTSTRAP_ROWS_PER_CLASS:', MAX_BOOTSTRAP_ROWS_PER_CLASS)

## 2. Utility Functions

In [ ]:
def now():
    return datetime.now().isoformat(timespec='seconds')

def write_csv_atomic(path, df):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    df.to_csv(tmp, index=False)
    tmp.replace(path)

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    tmp.replace(path)

def append_log(message):
    line = f'[{now()}] {message}'
    print(line)
    with open(LOGS_DIR / 'bootstrap_alert_profile.log', 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def ece_score(y_true, scores, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ids = np.digitize(scores, bins[1:-1], right=True)
    ece = 0.0
    n = len(y_true)
    for b in range(n_bins):
        mask = ids == b
        if not np.any(mask):
            continue
        conf = float(np.mean(scores[mask]))
        acc = float(np.mean(y_true[mask]))
        ece += (np.sum(mask) / n) * abs(acc - conf)
    return float(ece)

def metric_values(y_true, scores, threshold):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores).astype(float)
    pred = (scores >= float(threshold)).astype(int)
    n = len(y_true)
    fp = int(np.sum((pred == 1) & (y_true == 0)))
    fn = int(np.sum((pred == 0) & (y_true == 1)))
    out = {
        'average_precision': float(average_precision_score(y_true, scores)),
        'mcc': float(matthews_corrcoef(y_true, pred)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'brier': float(brier_score_loss(y_true, scores)),
        'ece': ece_score(y_true, scores),
        'cost_per_10k': float((FP_COST * fp + FN_COST * fn) / n * 10_000),
    }
    return out

def safe_name(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('_')

def scenario_axis_from_path(path):
    text = str(path).replace('\\', '/').lower()
    if 'leakage_stress_multiseed' in text or '03_leakage_stress_testing' in text:
        return 'leakage'
    if '06_temporal_robustness' in text:
        return 'temporal'
    if '07_prevalence_stress_test' in text:
        return 'prevalence'
    if '08_calibration_recalibration' in text:
        return 'calibration'
    if '09_explanation_stability' in text:
        return 'explanation'
    return 'unknown'

## 3. Locate Prediction Export Files

In [ ]:
REQUIRED_COLUMNS = ['dataset', 'model', 'scenario', 'seed', 'split', 'row_id', 'y_true', 'score', 'threshold']

all_pred_files = sorted(PROJECT_ROOT.rglob('*prediction_exports/*predictions.csv'))

# Prefer multiseed leakage over legacy single-seed leakage if both exist.
has_multiseed_leakage = any('leakage_stress_multiseed' in str(p).lower() for p in all_pred_files)
pred_files = []
for path in all_pred_files:
    text = str(path).lower().replace('\\', '/')
    if has_multiseed_leakage and 'leakage_stress_v1/prediction_exports' in text and 'multiseed' not in text:
        continue
    if 'patched_notebooks' in text:
        continue
    pred_files.append(path)

manifest_rows = []
bad_files = []
for path in pred_files:
    try:
        preview = pd.read_csv(path, nrows=5)
    except Exception as exc:
        bad_files.append({'path': str(path), 'error': repr(exc)})
        continue
    missing = [c for c in REQUIRED_COLUMNS if c not in preview.columns]
    if missing:
        bad_files.append({'path': str(path), 'error': f'missing columns: {missing}'})
        continue
    one = preview.iloc[0]
    manifest_rows.append({
        'axis': scenario_axis_from_path(path),
        'dataset': str(one['dataset']),
        'model': str(one['model']),
        'scenario': str(one['scenario']),
        'seed': int(one['seed']),
        'split': str(one['split']),
        'path': str(path),
    })

manifest = pd.DataFrame(manifest_rows).drop_duplicates()
write_csv_atomic(RESULTS_DIR / 'prediction_export_manifest.csv', manifest)
write_json(RESULTS_DIR / 'prediction_export_bad_files.json', bad_files)

print('Prediction files accepted:', len(manifest))
print('Bad files:', len(bad_files))
display(manifest.groupby(['axis', 'scenario']).size().reset_index(name='n_files').sort_values(['axis', 'scenario']))

## 4. Build Comparison Plan

In [ ]:
def path_for(axis, dataset, model, scenario, seed):
    rows = manifest[
        (manifest['axis'] == axis) &
        (manifest['dataset'] == dataset) &
        (manifest['model'] == model) &
        (manifest['scenario'] == scenario) &
        (manifest['seed'] == int(seed))
    ]
    if len(rows) == 0:
        return None
    return Path(rows.iloc[0]['path'])

comparison_rows = []
for seed in sorted(manifest['seed'].unique()):
    for dataset in sorted(manifest['dataset'].unique()):
        for model in sorted(manifest['model'].unique()):
            # Leakage axis.
            if dataset in ['D1_creditcard', 'D2_fraudTest']:
                ref, pert = 'SAFE_STRICT', 'UNSAFE_SMOTE_BEFORE_SPLIT'
            else:
                ref, pert = 'D3_STRICT_SAFE', 'D3_LEDGER_STATE_SAFE'
            if path_for('leakage', dataset, model, ref, seed) and path_for('leakage', dataset, model, pert, seed):
                comparison_rows.append({
                    'audit_axis': 'leakage',
                    'dataset': dataset,
                    'model': model,
                    'seed': seed,
                    'reference_scenario': ref,
                    'perturbed_scenario': pert,
                    'changed_assumption': 'feature availability / unsafe resampling',
                    'artifact_id': 'M03-LII',
                })

            # Temporal axis.
            if path_for('temporal', dataset, model, 'RANDOM_STRATIFIED_SAFE', seed) and path_for('temporal', dataset, model, 'CHRONOLOGICAL_SAFE', seed):
                comparison_rows.append({
                    'audit_axis': 'temporal',
                    'dataset': dataset,
                    'model': model,
                    'seed': seed,
                    'reference_scenario': 'RANDOM_STRATIFIED_SAFE',
                    'perturbed_scenario': 'CHRONOLOGICAL_SAFE',
                    'changed_assumption': 'temporal ordering',
                    'artifact_id': 'M06-TRD',
                })

            # Calibration axis.
            for pert in ['CALIBRATION_PLATT_SIGMOID', 'CALIBRATION_ISOTONIC']:
                if path_for('calibration', dataset, model, 'CALIBRATION_RAW', seed) and path_for('calibration', dataset, model, pert, seed):
                    comparison_rows.append({
                        'audit_axis': 'calibration',
                        'dataset': dataset,
                        'model': model,
                        'seed': seed,
                        'reference_scenario': 'CALIBRATION_RAW',
                        'perturbed_scenario': pert,
                        'changed_assumption': 'validation-fitted probability recalibration',
                        'artifact_id': 'M08-CAL',
                    })

            # Explanation/protocol predictive side; explanation ranks are merged later from existing result CSVs.
            if path_for('explanation', dataset, model, 'RANDOM_STRATIFIED_SAFE', seed) and path_for('explanation', dataset, model, 'CHRONOLOGICAL_SAFE', seed):
                comparison_rows.append({
                    'audit_axis': 'explanation_protocol',
                    'dataset': dataset,
                    'model': model,
                    'seed': seed,
                    'reference_scenario': 'RANDOM_STRATIFIED_SAFE',
                    'perturbed_scenario': 'CHRONOLOGICAL_SAFE',
                    'changed_assumption': 'explanation protocol split',
                    'artifact_id': 'M09-EII',
                })

comparison_plan = pd.DataFrame(comparison_rows)
write_csv_atomic(RESULTS_DIR / 'bootstrap_comparison_plan.csv', comparison_plan)
print('Comparison rows:', len(comparison_plan))
display(comparison_plan.groupby(['audit_axis', 'reference_scenario', 'perturbed_scenario']).size().reset_index(name='n'))

## 5. Bootstrap Functions

In [ ]:
def load_prediction(path):
    cols = REQUIRED_COLUMNS
    df = pd.read_csv(path, usecols=cols)
    df = df.sort_values('row_id').reset_index(drop=True)
    df['y_true'] = df['y_true'].astype(int)
    df['score'] = df['score'].astype(float).clip(0, 1)
    df['threshold'] = df['threshold'].astype(float)
    return df

def maybe_cap_by_class(df, rng):
    if MAX_BOOTSTRAP_ROWS_PER_CLASS is None:
        return df.reset_index(drop=True)
    parts = []
    for label in [0, 1]:
        part = df[df['y_true'] == label]
        if len(part) > MAX_BOOTSTRAP_ROWS_PER_CLASS:
            take = rng.choice(part.index.values, size=MAX_BOOTSTRAP_ROWS_PER_CLASS, replace=False)
            part = part.loc[take]
        parts.append(part)
    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=int(rng.integers(1, 2**31 - 1))).reset_index(drop=True)

def align_for_bootstrap(ref_df, pert_df):
    ref_key = ref_df[['row_id', 'y_true', 'score', 'threshold']].rename(columns={'score': 'score_ref', 'threshold': 'threshold_ref'})
    pert_key = pert_df[['row_id', 'y_true', 'score', 'threshold']].rename(columns={'score': 'score_pert', 'threshold': 'threshold_pert'})
    merged = ref_key.merge(pert_key, on=['row_id', 'y_true'], how='inner')
    paired = len(merged) == len(ref_df) == len(pert_df)
    if paired:
        return 'paired', merged.reset_index(drop=True), None, None
    return 'stratified_unpaired', None, ref_df.reset_index(drop=True), pert_df.reset_index(drop=True)

def stratified_indices(y, rng):
    y = np.asarray(y).astype(int)
    out = []
    for label in [0, 1]:
        idx = np.flatnonzero(y == label)
        if len(idx) == 0:
            continue
        out.append(rng.choice(idx, size=len(idx), replace=True))
    if not out:
        return np.array([], dtype=int)
    return np.concatenate(out)

def bootstrap_one_metric(y_ref, s_ref, t_ref, y_pert, s_pert, t_pert, metric, rng, mode):
    values = []
    if mode == 'paired':
        y = y_ref
        for _ in range(N_BOOTSTRAP):
            idx = stratified_indices(y, rng)
            if len(np.unique(y[idx])) < 2:
                continue
            a = metric_values(y[idx], s_ref[idx], t_ref)[metric]
            b = metric_values(y[idx], s_pert[idx], t_pert)[metric]
            values.append(b - a)
    else:
        for _ in range(N_BOOTSTRAP):
            idx_ref = stratified_indices(y_ref, rng)
            idx_pert = stratified_indices(y_pert, rng)
            if len(np.unique(y_ref[idx_ref])) < 2 or len(np.unique(y_pert[idx_pert])) < 2:
                continue
            a = metric_values(y_ref[idx_ref], s_ref[idx_ref], t_ref)[metric]
            b = metric_values(y_pert[idx_pert], s_pert[idx_pert], t_pert)[metric]
            values.append(b - a)
    return np.asarray(values, dtype=float)

def cost_threshold_for_reference(metric, ref_value):
    if metric != 'cost_per_10k':
        return PRACTICAL_THRESHOLDS.get(metric)
    return max(100.0, abs(float(ref_value)) * 0.10)

def interpret_delta(metric, delta, ci_low, ci_high, ref_value):
    threshold = cost_threshold_for_reference(metric, ref_value)
    if threshold is None or pd.isna(ci_low) or pd.isna(ci_high):
        return 'inconclusive'
    low_abs_ok = abs(ci_low) < threshold and abs(ci_high) < threshold
    separated_material = (ci_low > threshold and ci_high > threshold) or (ci_low < -threshold and ci_high < -threshold)
    if low_abs_ok:
        return 'robust'
    if separated_material:
        return 'fragile'
    return 'inconclusive'

def checkpoint_path_for(row):
    name = '__'.join([
        safe_name(row['audit_axis']),
        safe_name(row['dataset']),
        safe_name(row['model']),
        f"seed{int(row['seed'])}",
        safe_name(row['reference_scenario']),
        safe_name(row['perturbed_scenario']),
    ]) + '.csv'
    return CHECKPOINT_DIR / name

## 6. Run Bootstrap With Resume Checkpoints

In [ ]:
bootstrap_rows = []
rng_master = np.random.default_rng(RANDOM_SEED)

for _, row in comparison_plan.iterrows():
    ckpt = checkpoint_path_for(row)
    if ckpt.exists():
        bootstrap_rows.append(pd.read_csv(ckpt))
        append_log(f'SKIP bootstrap checkpoint: {ckpt.name}')
        continue

    axis_for_path = row['audit_axis']
    if axis_for_path == 'explanation_protocol':
        axis_for_path = 'explanation'
    ref_path = path_for(axis_for_path, row['dataset'], row['model'], row['reference_scenario'], row['seed'])
    pert_path = path_for(axis_for_path, row['dataset'], row['model'], row['perturbed_scenario'], row['seed'])
    append_log(f'BOOTSTRAP {row["audit_axis"]} {row["dataset"]} {row["model"]} seed={row["seed"]}: {row["reference_scenario"]} -> {row["perturbed_scenario"]}')

    ref_df = load_prediction(ref_path)
    pert_df = load_prediction(pert_path)
    rng = np.random.default_rng(int(rng_master.integers(1, 2**31 - 1)))
    ref_df = maybe_cap_by_class(ref_df, rng)
    pert_df = maybe_cap_by_class(pert_df, rng)

    mode, merged, ref_unpaired, pert_unpaired = align_for_bootstrap(ref_df, pert_df)
    if mode == 'paired':
        y_ref = merged['y_true'].to_numpy()
        y_pert = y_ref
        s_ref = merged['score_ref'].to_numpy()
        s_pert = merged['score_pert'].to_numpy()
        t_ref = float(merged['threshold_ref'].iloc[0])
        t_pert = float(merged['threshold_pert'].iloc[0])
        n_ref = n_pert = len(merged)
    else:
        y_ref = ref_unpaired['y_true'].to_numpy()
        y_pert = pert_unpaired['y_true'].to_numpy()
        s_ref = ref_unpaired['score'].to_numpy()
        s_pert = pert_unpaired['score'].to_numpy()
        t_ref = float(ref_unpaired['threshold'].iloc[0])
        t_pert = float(pert_unpaired['threshold'].iloc[0])
        n_ref = len(ref_unpaired)
        n_pert = len(pert_unpaired)

    ref_metrics = metric_values(y_ref, s_ref, t_ref)
    pert_metrics = metric_values(y_pert, s_pert, t_pert)

    out_rows = []
    for metric in METRICS:
        samples = bootstrap_one_metric(y_ref, s_ref, t_ref, y_pert, s_pert, t_pert, metric, rng, mode)
        delta = float(pert_metrics[metric] - ref_metrics[metric])
        ci_low, ci_high = np.percentile(samples, [2.5, 97.5]) if len(samples) else (np.nan, np.nan)
        out_rows.append({
            'audit_axis': row['audit_axis'],
            'dataset': row['dataset'],
            'model': row['model'],
            'seed': int(row['seed']),
            'reference_scenario': row['reference_scenario'],
            'perturbed_scenario': row['perturbed_scenario'],
            'changed_assumption': row['changed_assumption'],
            'metric': metric,
            'reference_value': ref_metrics[metric],
            'perturbed_value': pert_metrics[metric],
            'delta': delta,
            'mean_delta': float(np.mean(samples)) if len(samples) else np.nan,
            'std_delta': float(np.std(samples, ddof=1)) if len(samples) > 1 else np.nan,
            'ci_low': float(ci_low),
            'ci_high': float(ci_high),
            'n_bootstrap': int(len(samples)),
            'bootstrap_mode': mode,
            'n_reference': int(n_ref),
            'n_perturbed': int(n_pert),
            'interpretation': interpret_delta(metric, delta, ci_low, ci_high, ref_metrics[metric]),
            'artifact_id': row['artifact_id'],
        })
    ckpt_df = pd.DataFrame(out_rows)
    write_csv_atomic(ckpt, ckpt_df)
    bootstrap_rows.append(ckpt_df)

bootstrap_per_seed = pd.concat(bootstrap_rows, ignore_index=True) if bootstrap_rows else pd.DataFrame()
write_csv_atomic(RESULTS_DIR / 'paired_bootstrap_delta_per_seed.csv', bootstrap_per_seed)
display(bootstrap_per_seed.head())
print('Rows:', len(bootstrap_per_seed))

## 7. Aggregate Bootstrap Results

In [ ]:
group_cols = ['audit_axis', 'dataset', 'model', 'reference_scenario', 'perturbed_scenario', 'changed_assumption', 'metric', 'artifact_id']

def summarize_group(group):
    deltas = group['delta'].astype(float)
    return pd.Series({
        'n_seeds': int(group['seed'].nunique()),
        'delta_mean_across_seeds': float(deltas.mean()),
        'delta_std_across_seeds': float(deltas.std(ddof=1)) if len(deltas) > 1 else 0.0,
        'ci_low_median': float(group['ci_low'].median()),
        'ci_high_median': float(group['ci_high'].median()),
        'n_bootstrap_min': int(group['n_bootstrap'].min()),
        'bootstrap_modes': ','.join(sorted(group['bootstrap_mode'].unique())),
        'interpretation_counts': json.dumps(group['interpretation'].value_counts().to_dict(), sort_keys=True),
    })

bootstrap_summary = bootstrap_per_seed.groupby(group_cols, dropna=False).apply(summarize_group).reset_index()
write_csv_atomic(RESULTS_DIR / 'paired_bootstrap_delta_summary.csv', bootstrap_summary)
display(bootstrap_summary.head(20))

## 8. Fixed Alert Budget Evaluation

In [ ]:
def alert_budget_metrics(y_true, scores, budget_count):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores).astype(float)
    n = len(y_true)
    budget_count = int(max(1, min(budget_count, n)))
    order = np.argsort(-scores, kind='mergesort')
    chosen = order[:budget_count]
    selected_y = y_true[chosen]
    tp = int(np.sum(selected_y == 1))
    fp = int(np.sum(selected_y == 0))
    fraud_total = int(np.sum(y_true == 1))
    fn = fraud_total - tp
    return {
        'n_eval': int(n),
        'budget_count': int(budget_count),
        'fraud_total': fraud_total,
        'fraud_captured': tp,
        'false_alerts': fp,
        'alert_precision': float(tp / budget_count) if budget_count else 0.0,
        'alert_recall': float(tp / fraud_total) if fraud_total else 0.0,
        'alert_cost_per_10k': float((FP_COST * fp + FN_COST * fn) / n * 10_000),
    }

alert_rows = []
for _, item in manifest.iterrows():
    ckpt = ALERT_CHECKPOINT_DIR / (safe_name(f"{item['axis']}__{item['dataset']}__{item['model']}__{item['scenario']}__seed{item['seed']}") + '.csv')
    if ckpt.exists():
        alert_rows.append(pd.read_csv(ckpt))
        continue
    df = load_prediction(item['path'])
    rows = []
    n = len(df)
    for budget in ALERT_BUDGETS_PER_10K:
        budget_count = max(1, int(round(n * budget / 10_000)))
        m = alert_budget_metrics(df['y_true'], df['score'], budget_count)
        rows.append({
            **item[['axis', 'dataset', 'model', 'scenario', 'seed']].to_dict(),
            'budget_type': 'alerts_per_10k',
            'budget_value': budget,
            **m,
        })
    for frac in TOP_FRACTION_BUDGETS:
        budget_count = max(1, int(round(n * frac)))
        m = alert_budget_metrics(df['y_true'], df['score'], budget_count)
        rows.append({
            **item[['axis', 'dataset', 'model', 'scenario', 'seed']].to_dict(),
            'budget_type': 'top_fraction',
            'budget_value': frac,
            **m,
        })
    ckpt_df = pd.DataFrame(rows)
    write_csv_atomic(ckpt, ckpt_df)
    alert_rows.append(ckpt_df)

alert_per_seed = pd.concat(alert_rows, ignore_index=True) if alert_rows else pd.DataFrame()
write_csv_atomic(RESULTS_DIR / 'alert_budget_per_seed.csv', alert_per_seed)

alert_summary = (
    alert_per_seed
    .groupby(['axis', 'dataset', 'model', 'scenario', 'budget_type', 'budget_value'], dropna=False)
    .agg(
        n_seeds=('seed', 'nunique'),
        precision_mean=('alert_precision', 'mean'),
        precision_std=('alert_precision', 'std'),
        recall_mean=('alert_recall', 'mean'),
        recall_std=('alert_recall', 'std'),
        false_alerts_mean=('false_alerts', 'mean'),
        cost_mean=('alert_cost_per_10k', 'mean'),
    )
    .reset_index()
)
write_csv_atomic(RESULTS_DIR / 'alert_budget_summary.csv', alert_summary)
display(alert_summary.head(20))

## 9. Build LC-ORF Audit Profile

In [ ]:
profile = bootstrap_summary.copy()
profile['profile_component'] = profile['audit_axis'] + ':' + profile['metric']
profile['delta_display'] = profile['delta_mean_across_seeds']
profile['ci_low'] = profile['ci_low_median']
profile['ci_high'] = profile['ci_high_median']
profile['evidence_status'] = np.where(
    profile['n_bootstrap_min'] >= N_BOOTSTRAP,
    'complete',
    'incomplete_bootstrap'
)

write_csv_atomic(RESULTS_DIR / 'lcorf_audit_profile.csv', profile)

heat_source = profile[profile['metric'].isin(['average_precision', 'mcc', 'brier', 'ece', 'cost_per_10k'])].copy()
heat_source['comparison'] = heat_source['audit_axis'] + '\\n' + heat_source['reference_scenario'] + ' -> ' + heat_source['perturbed_scenario'] + '\\n' + heat_source['metric']
heat = heat_source.pivot_table(
    index=['dataset', 'model'],
    columns='comparison',
    values='delta_display',
    aggfunc='mean',
)

plt.figure(figsize=(max(10, 0.45 * len(heat.columns)), max(4, 0.45 * len(heat))))
ax = sns.heatmap(heat, center=0, cmap='vlag', linewidths=0.35, annot=False)
ax.set_title('LC-ORF Counterfactual Audit Profile: Mean Delta Across Seeds')
ax.set_xlabel('')
ax.set_ylabel('Dataset / model')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
png = FIGURES_DIR / 'lcorf_audit_profile_heatmap.png'
pdf = FIGURES_DIR / 'lcorf_audit_profile_heatmap.pdf'
plt.savefig(png, dpi=600, bbox_inches='tight')
plt.savefig(pdf, bbox_inches='tight')
plt.show()

print('Saved:', RESULTS_DIR / 'lcorf_audit_profile.csv')
print('Saved:', png)

## 10. Alert Budget Figure

In [ ]:
plot_df = alert_summary[
    (alert_summary['budget_type'] == 'alerts_per_10k') &
    (alert_summary['budget_value'].isin(ALERT_BUDGETS_PER_10K))
].copy()

if len(plot_df):
    g = sns.catplot(
        data=plot_df,
        x='budget_value',
        y='precision_mean',
        hue='model',
        col='dataset',
        row='axis',
        kind='bar',
        height=2.6,
        aspect=1.25,
        sharey=False,
    )
    g.set_axis_labels('Alerts per 10,000 transactions', 'Precision at alert budget')
    g.set_titles('{row_name} / {col_name}')
    plt.tight_layout()
    png = FIGURES_DIR / 'alert_budget_precision_by_axis_dataset.png'
    pdf = FIGURES_DIR / 'alert_budget_precision_by_axis_dataset.pdf'
    plt.savefig(png, dpi=600, bbox_inches='tight')
    plt.savefig(pdf, bbox_inches='tight')
    plt.show()
    print('Saved:', png)

## 11. Run Summary and Final Checks

In [ ]:
summary = {
    'completed_at': now(),
    'project_root': str(PROJECT_ROOT),
    'n_prediction_files': int(len(manifest)),
    'n_comparisons': int(len(comparison_plan)),
    'n_bootstrap': int(N_BOOTSTRAP),
    'max_bootstrap_rows_per_class': MAX_BOOTSTRAP_ROWS_PER_CLASS,
    'bootstrap_per_seed_file': str(RESULTS_DIR / 'paired_bootstrap_delta_per_seed.csv'),
    'bootstrap_summary_file': str(RESULTS_DIR / 'paired_bootstrap_delta_summary.csv'),
    'alert_budget_per_seed_file': str(RESULTS_DIR / 'alert_budget_per_seed.csv'),
    'alert_budget_summary_file': str(RESULTS_DIR / 'alert_budget_summary.csv'),
    'audit_profile_file': str(RESULTS_DIR / 'lcorf_audit_profile.csv'),
    'figures_dir': str(FIGURES_DIR),
    'bootstrap_modes': sorted(bootstrap_per_seed['bootstrap_mode'].unique().tolist()) if len(bootstrap_per_seed) else [],
    'min_bootstrap_replicates': int(bootstrap_per_seed['n_bootstrap'].min()) if len(bootstrap_per_seed) else 0,
}
write_json(RESULTS_DIR / 'lcorf_bootstrap_alert_profile_run_summary.json', summary)
print(json.dumps(summary, indent=2))

assert len(manifest) > 0, 'No prediction exports found.'
assert len(comparison_plan) > 0, 'No comparisons created.'
assert len(bootstrap_per_seed) > 0, 'No bootstrap rows saved.'
assert bootstrap_per_seed['n_bootstrap'].min() >= N_BOOTSTRAP, 'Some bootstrap rows have fewer replicates than requested.'
assert (RESULTS_DIR / 'lcorf_audit_profile.csv').exists(), 'Audit profile missing.'